In [7]:
import pandas as pd
import requests
import pandas as pd

url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB4441/data"

params = {
    "lang": "en",
    "valueCodes[Region]": "*",
    "valueCodes[Yrke2012]": "*",
    "valueCodes[Alder]": "*",
    "valueCodes[Kon]": "*",
    "valueCodes[Tid]": "2024",
    "valueCodes[ContentsCode]": "000006Y0"
}

response = requests.get(url, params=params)

print("Status:", response.status_code)

location_data = response.json()

print(location_data.keys())



Status: 200
dict_keys(['version', 'class', 'label', 'source', 'updated', 'note', 'role', 'id', 'size', 'dimension', 'extension', 'value'])


In [8]:
dimensions = location_data["id"]
categories = location_data["dimension"]
sizes = location_data["size"]
values = location_data["value"]

position_to_code = {}

for dimension in dimensions:
    index = categories[dimension]["category"]["index"]
    position_to_code[dimension] = {
        position: code
        for code, position in index.items()
    }

# Create rows
rows = []

for flat_position, value in enumerate(values):

    # Convert the flat position into a position for each dimension
    positions = []
    remainder = flat_position

    for size in reversed(sizes):
        positions.append(remainder % size)
        remainder = remainder // size

    positions = list(reversed(positions))

    row = {}

    for dimension, position in zip(dimensions, positions):
        code = position_to_code[dimension][position]
        label = categories[dimension]["category"]["label"][code]

        row[dimension] = label

    row["Employees"] = value

    rows.append(row)

location_df = pd.DataFrame(rows)

location_df.head()
print(location_df.shape)

(65560, 7)


In [9]:
location_df.head()

,Region,Yrke2012,Alder,Kon,ContentsCode,Tid,Employees
0,Sweden,Commissioned armed forces officers,16–24 years,men,Number,2024,60
1,Sweden,Commissioned armed forces officers,16–24 years,women,Number,2024,31
2,Sweden,Commissioned armed forces officers,25–29 years,men,Number,2024,102
3,Sweden,Commissioned armed forces officers,25–29 years,women,Number,2024,40
4,Sweden,Commissioned armed forces officers,30–34 years,men,Number,2024,232


In [10]:
location_df = pd.DataFrame(rows)